# Model 2: Pretrained MiniLM/DistilBERT Batched Pairwise Ranker

**Project:** Smart MCQ Solver · DL & GenAI · Milestone 3
**Kernel path:** `nb/pretrained/pre_trained.ipynb`

This notebook uses a fully vectorized, batched dual-encoder approach for rapid execution.

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import AutoTokenizer, AutoModel
import wandb
import warnings
warnings.filterwarnings('ignore')

try:
    from kaggle_secrets import UserSecretsClient
    user_secrets = UserSecretsClient()
    wandb_key = user_secrets.get_secret("wandb_api_key")
except:
    # Fallback to hardcoded key for local/testing
    wandb_key = "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"


In [ ]:
KAGGLE_INPUT = "/kaggle/input/competitions/smart-mcq-solver-challenge"

def get_path(filename: str) -> str:
    candidates = [
        os.path.join("..", "..", "data", filename),
        os.path.join("data", filename),
        os.path.join(KAGGLE_INPUT, filename),
    ]
    for p in candidates:
        if os.path.exists(p):
            return p
    return filename

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")


In [ ]:
wandb.login(key=wandb_key, relogin=True)
run = wandb.init(
    project="23f2004343-t22026",
    name="Model_2_Pretrained_Optimized",
    config={
        "base_model": "sentence-transformers/all-MiniLM-L6-v2",
        "max_seq_length": 128,
        "batch_size": 32,
        "learning_rate": 2e-5,
        "epochs": 2
    }
)
cfg = wandb.config


In [ ]:
trn_df = pd.read_csv(get_path("train.csv"))
tst_df = pd.read_csv(get_path("test.csv"))
tokenizer = AutoTokenizer.from_pretrained(cfg.base_model)

class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_len, is_train=True):
        self.df = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.is_train = is_train
        self.options = ['A', 'B', 'C', 'D', 'E']
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        prompt = str(row['prompt'])
        
        # Pre-tokenize prompt
        prompt_enc = self.tokenizer(
            prompt, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt"
        )
        
        # Pre-tokenize options
        options_texts = [str(row[opt]) for opt in self.options]
        options_enc = self.tokenizer(
            options_texts, max_length=self.max_len, padding="max_length", truncation=True, return_tensors="pt"
        )
        
        item = {
            "prompt_ids": prompt_enc["input_ids"].squeeze(0),
            "prompt_mask": prompt_enc["attention_mask"].squeeze(0),
            "options_ids": options_enc["input_ids"],      # shape: (5, seq_len)
            "options_mask": options_enc["attention_mask"] # shape: (5, seq_len)
        }
        
        if self.is_train:
            ans = row['answer']
            label_idx = self.options.index(ans) if ans in self.options else 0
            labels = torch.zeros(5)
            labels[label_idx] = 1.0
            item["labels"] = labels
            
        return item


In [ ]:
from sklearn.model_selection import train_test_split
trn_sub, val_sub = train_test_split(trn_df, test_size=0.1, random_state=42)

trn_ds = MCQDataset(trn_sub, tokenizer, cfg.max_seq_length, is_train=True)
val_ds = MCQDataset(val_sub, tokenizer, cfg.max_seq_length, is_train=True)
tst_ds = MCQDataset(tst_df, tokenizer, cfg.max_seq_length, is_train=False)

trn_loader = DataLoader(trn_ds, batch_size=cfg.batch_size, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)
tst_loader = DataLoader(tst_ds, batch_size=cfg.batch_size, shuffle=False, num_workers=2, pin_memory=True)


In [ ]:
class DualEncoderRanker(nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(model_name)
        
    def mean_pooling(self, model_output, attention_mask):
        token_embeddings = model_output[0]
        input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
        return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)

    def forward(self, prompt_ids, prompt_mask, options_ids, options_mask):
        # prompt_ids: (batch, seq_len)
        prompt_out = self.encoder(input_ids=prompt_ids, attention_mask=prompt_mask)
        prompt_emb = self.mean_pooling(prompt_out, prompt_mask) # (batch, hidden_dim)
        
        # options_ids: (batch, 5, seq_len) -> reshape to (batch*5, seq_len)
        B, N, S = options_ids.shape
        opt_ids_flat = options_ids.view(B*N, S)
        opt_mask_flat = options_mask.view(B*N, S)
        
        opt_out = self.encoder(input_ids=opt_ids_flat, attention_mask=opt_mask_flat)
        opt_emb_flat = self.mean_pooling(opt_out, opt_mask_flat) # (batch*5, hidden_dim)
        opt_emb = opt_emb_flat.view(B, N, -1) # (batch, 5, hidden_dim)
        
        # Cosine similarity between prompt and options
        # prompt_emb: (batch, hidden), opt_emb: (batch, 5, hidden)
        prompt_emb_norm = F.normalize(prompt_emb, p=2, dim=-1)
        opt_emb_norm = F.normalize(opt_emb, p=2, dim=-1)
        
        # batch matrix multiplication: (B, 1, H) x (B, H, 5) -> (B, 1, 5)
        logits = torch.bmm(prompt_emb_norm.unsqueeze(1), opt_emb_norm.transpose(1, 2)).squeeze(1) # (B, 5)
        # Scale logits for BCE (optional, usually similarity is -1 to 1, we can multiply by a temperature)
        logits = logits * 20.0 
        return logits

model = DualEncoderRanker(cfg.base_model).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=cfg.learning_rate)
criterion = nn.BCEWithLogitsLoss()


In [ ]:
from sklearn.metrics import accuracy_score, f1_score

for epoch in range(1, cfg.epochs + 1):
    model.train()
    trn_loss = 0.0
    for batch in trn_loader:
        optimizer.zero_grad()
        logits = model(
            batch["prompt_ids"].to(DEVICE),
            batch["prompt_mask"].to(DEVICE),
            batch["options_ids"].to(DEVICE),
            batch["options_mask"].to(DEVICE)
        )
        loss = criterion(logits, batch["labels"].to(DEVICE))
        loss.backward()
        optimizer.step()
        trn_loss += loss.item()
        
    # Eval
    model.eval()
    val_loss, all_preds, all_labels = 0.0, [], []
    with torch.no_grad():
        for batch in val_loader:
            logits = model(
                batch["prompt_ids"].to(DEVICE),
                batch["prompt_mask"].to(DEVICE),
                batch["options_ids"].to(DEVICE),
                batch["options_mask"].to(DEVICE)
            )
            labels = batch["labels"].to(DEVICE)
            loss = criterion(logits, labels)
            val_loss += loss.item()
            
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            true_labels = torch.argmax(labels, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(true_labels)
            
    val_acc = accuracy_score(all_labels, all_preds)
    val_f1 = f1_score(all_labels, all_preds, average='macro')
    avg_trn_loss = trn_loss / len(trn_loader)
    avg_val_loss = val_loss / len(val_loader)
    
    wandb.log({"epoch": epoch, "train_loss": avg_trn_loss, "val_loss": avg_val_loss, "val_acc": val_acc, "val_f1": val_f1})
    print(f"Epoch {epoch}: Train Loss {avg_trn_loss:.4f} | Val Loss {avg_val_loss:.4f} | Val Acc {val_acc:.4f} | Val F1 {val_f1:.4f}")


In [ ]:
model.eval()
options_letters = np.array(['A', 'B', 'C', 'D', 'E'])
all_top3 = []

print("Running batched inference on test set...")
with torch.no_grad():
    for batch in tst_loader:
        logits = model(
            batch["prompt_ids"].to(DEVICE),
            batch["prompt_mask"].to(DEVICE),
            batch["options_ids"].to(DEVICE),
            batch["options_mask"].to(DEVICE)
        )
        # logits shape: (batch, 5)
        top3_idx = torch.topk(logits, 3, dim=1).indices.cpu().numpy()
        for idx_row in top3_idx:
            all_top3.append(" ".join(options_letters[idx_row]))

sub_df = pd.DataFrame({'id': tst_df['id'], 'prediction': all_top3})
sub_df.to_csv("submission.csv", index=False)
print(f"Exported submission.csv with {len(sub_df)} rows.")

art = wandb.Artifact("submission_pretrained_optimized", type="predictions")
art.add_file("submission.csv")
wandb.log_artifact(art)
wandb.finish()
